# Loading Dataset

In [1]:
# ============================================================
# DAY 19 - CONTROLLED HYPERPARAMETER / PARAMETER EXPERIMENTS
# Models: LSTM, GRU, TCN
# ============================================================

import os
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# -----------------------------
# Training configuration
# -----------------------------
BATCH_SIZE = 32
EPOCHS = 30
BASE_LR = 0.001

print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Base learning rate:", BASE_LR)

Device: cpu
Batch size: 32
Epochs: 30
Base learning rate: 0.001


In [3]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ============================================================
# FIND sequences_v1.npz IN CURRENT GOOGLE DRIVE
# ============================================================

import os

target = "sequences_v1.npz"
matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if target in files:
        matches.append(
            os.path.join(root, target)
        )

print("Search completed.\n")

if matches:
    print("FOUND:")
    for path in matches:
        print("✅", path)
else:
    print("❌ sequences_v1.npz was NOT found in MyDrive")

Search completed.

FOUND:
✅ /content/drive/MyDrive/IITG dataset/cleaned_dataset/processed_dataset_v1/sequences_v1.npz


In [11]:
# ============================================================
# LOAD DAY 18 DATASET
# ============================================================

import numpy as np

DATA_PATH = "/content/drive/MyDrive/IITG dataset/cleaned_dataset/processed_dataset_v1/sequences_v1.npz"

data = np.load(DATA_PATH)

print("Available arrays:")
print(data.files)

print("\nArray shapes:")
for key in data.files:
    print(key, data[key].shape)

Available arrays:
['X', 'SoH', 'RUL', 'battery_id', 'split']

Array shapes:
X (2287, 10, 2)
SoH (2287,)
RUL (2287,)
battery_id (2287,)
split (2287,)


In [12]:
# ============================================================
# DAY 19 - CHECK EXISTING DATA SPLITS
# ============================================================

X = data["X"]
SoH = data["SoH"]
RUL = data["RUL"]
battery_id = data["battery_id"]
split = data["split"]

print("X shape:", X.shape)
print("SoH shape:", SoH.shape)
print("RUL shape:", RUL.shape)
print("Battery IDs:", battery_id.shape)
print("Split:", split.shape)

print("\nUnique split values:")
print(np.unique(split, return_counts=True))

X shape: (2287, 10, 2)
SoH shape: (2287,)
RUL shape: (2287,)
Battery IDs: (2287,)
Split: (2287,)

Unique split values:
(array(['test', 'train', 'validation'], dtype='<U10'), array([ 392, 1516,  379]))


In [13]:
# ============================================================
# DAY 19 - CREATE TRAIN / VALIDATION / TEST SETS
# ============================================================

# Training data
train_idx = split == "train"

# Validation data
val_idx = split == "validation"

# Test data
test_idx = split == "test"

X_train = X[train_idx]
y_train = SoH[train_idx]

X_val = X[val_idx]
y_val = SoH[val_idx]

X_test = X[test_idx]
y_test = SoH[test_idx]

print("TRAIN")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nVALIDATION")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nTEST")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

TRAIN
X_train: (1516, 10, 2)
y_train: (1516,)

VALIDATION
X_val: (379, 10, 2)
y_val: (379,)

TEST
X_test: (392, 10, 2)
y_test: (392,)


In [14]:
# ============================================================
# DAY 19 - MODEL EXPERIMENT SETUP
# ============================================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
import pandas as pd
import time

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [16]:
# ============================================================
# COMMON EVALUATION FUNCTION
# ============================================================

def evaluate_model(model, X_data, y_data):

    start_time = time.time()

    predictions = model.predict(X_data, verbose=0).ravel()

    inference_time = time.time() - start_time

    mse = mean_squared_error(y_data, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_data, predictions)
    r2 = r2_score(y_data, predictions)

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Inference_Time_s": inference_time
    }

In [17]:
# ============================================================
# EXPERIMENT LOG
# ============================================================

experiment_results = []

def record_experiment(
    model_name,
    experiment_name,
    config,
    training_time,
    val_metrics
):

    experiment_results.append({
        "Model": model_name,
        "Experiment": experiment_name,
        "Configuration": str(config),
        "Training_Time_s": training_time,
        "Val_MSE": val_metrics["MSE"],
        "Val_RMSE": val_metrics["RMSE"],
        "Val_MAE": val_metrics["MAE"],
        "Val_R2": val_metrics["R2"],
        "Inference_Time_s": val_metrics["Inference_Time_s"]
    })

# LSTM

In [19]:
# ============================================================
# DAY 19 - LSTM BASELINE EXPERIMENT
# ============================================================

def build_lstm_model(
    units=64,
    dropout=0.0,
    learning_rate=0.001
):

    model = keras.Sequential([
        layers.Input(shape=(X_train.shape[1], X_train.shape[2])),

        layers.LSTM(units),

        layers.Dropout(dropout),

        layers.Dense(1)
    ])

    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

# HYPERPARAMETER TUNING

In [21]:
# ============================================================
# DAY 19 - CONTROLLED LSTM HYPERPARAMETER TUNING
# ============================================================

import time
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ------------------------------------------------------------
# Controlled LSTM experiments
# ------------------------------------------------------------

LSTM_EXPERIMENTS = [

    # Units, Learning Rate, Dropout, Batch Size

    (32,  0.0010, 0.0, 32),
    (32,  0.0005, 0.0, 32),

    (64,  0.0010, 0.0, 32),
    (64,  0.0005, 0.0, 32),
    (64,  0.0001, 0.0, 32),

    (128, 0.0010, 0.0, 32),
    (128, 0.0005, 0.0, 32),

    (64,  0.0010, 0.1, 16),
    (64,  0.0005, 0.1, 16),

    (128, 0.0005, 0.1, 16),
    (128, 0.0005, 0.2, 32),
    (128, 0.0001, 0.1, 32)
]

MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8

print("Total LSTM experiments:", len(LSTM_EXPERIMENTS))

Total LSTM experiments: 12


# GRID SEARCH

In [22]:
# ============================================================
# RUN CONTROLLED LSTM EXPERIMENTS
# ============================================================

lstm_results = []

RESULTS_FILE = "Day19_LSTM_Tuning_Results.csv"

for run_no, (units, learning_rate, dropout, batch_size) in enumerate(
    LSTM_EXPERIMENTS, start=1
):

    print("\n" + "="*70)
    print(f"LSTM Experiment {run_no}/{len(LSTM_EXPERIMENTS)}")
    print(f"Units        : {units}")
    print(f"Learning Rate: {learning_rate}")
    print(f"Dropout      : {dropout}")
    print(f"Batch Size   : {batch_size}")
    print("="*70)

    # --------------------------------------------------------
    # Clear previous TensorFlow model from memory
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    # Reproducibility
    tf.random.set_seed(42)
    np.random.seed(42)

    # --------------------------------------------------------
    # Build LSTM
    # --------------------------------------------------------

    model = build_lstm_model(
        units=units,
        dropout=dropout,
        learning_rate=learning_rate
    )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=0
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    start_train = time.time()

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        callbacks=[early_stopping],
        verbose=0
    )

    training_time = time.time() - start_train

    # --------------------------------------------------------
    # Validation prediction
    # --------------------------------------------------------

    y_val_pred = model.predict(
        X_val,
        verbose=0
    ).ravel()

    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------

    val_mse = mean_squared_error(
        y_val,
        y_val_pred
    )

    val_rmse = np.sqrt(val_mse)

    val_mae = mean_absolute_error(
        y_val,
        y_val_pred
    )

    val_r2 = r2_score(
        y_val,
        y_val_pred
    )

    # --------------------------------------------------------
    # Best epoch
    # --------------------------------------------------------

    best_epoch = np.argmin(
        history.history["val_loss"]
    ) + 1

    best_val_loss = np.min(
        history.history["val_loss"]
    )

    # --------------------------------------------------------
    # Save result
    # --------------------------------------------------------

    lstm_results.append({
        "Run": run_no,
        "Model": "LSTM",
        "Units": units,
        "Learning_Rate": learning_rate,
        "Dropout": dropout,
        "Batch_Size": batch_size,
        "Best_Epoch": best_epoch,
        "Best_Val_Loss": best_val_loss,
        "Val_MSE": val_mse,
        "Val_RMSE": val_rmse,
        "Val_MAE": val_mae,
        "Val_R2": val_r2,
        "Training_Time_s": training_time,
        "Parameters": model.count_params()
    })

    # Save after every experiment
    pd.DataFrame(lstm_results).to_csv(
        RESULTS_FILE,
        index=False
    )

    print(
        f"Val RMSE = {val_rmse:.6f} | "
        f"Val MAE = {val_mae:.6f} | "
        f"Val R² = {val_r2:.6f}"
    )

print("\nLSTM controlled experiments completed!")
print(f"Total experiments: {len(lstm_results)}")
print(f"Results saved to: {RESULTS_FILE}")


LSTM Experiment 1/12
Units        : 32
Learning Rate: 0.001
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 24.463276 | Val MAE = 17.969177 | Val R² = -0.023559

LSTM Experiment 2/12
Units        : 32
Learning Rate: 0.0005
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 34.728576 | Val MAE = 33.276662 | Val R² = -1.062803

LSTM Experiment 3/12
Units        : 64
Learning Rate: 0.001
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 24.180033 | Val MAE = 15.915537 | Val R² = 0.000006

LSTM Experiment 4/12
Units        : 64
Learning Rate: 0.0005
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 24.414312 | Val MAE = 17.753658 | Val R² = -0.019466

LSTM Experiment 5/12
Units        : 64
Learning Rate: 0.0001
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 47.932046 | Val MAE = 45.719770 | Val R² = -2.929486

LSTM Experiment 6/12
Units        : 128
Learning Rate: 0.001
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 24.180193 | Val MAE = 15.909536 | Val R² = -0.000008

LSTM Experiment 7/12
Uni

# BEST LSTM CONFIGURATION

In [23]:
lstm_df = pd.DataFrame(lstm_results)

# Sort by validation RMSE
lstm_df_sorted = lstm_df.sort_values(
    by="Val_RMSE",
    ascending=True
).reset_index(drop=True)

print("TOP 10 LSTM CONFIGURATIONS")
display(
    lstm_df_sorted.head(10)
)

TOP 10 LSTM CONFIGURATIONS


,Run,Model,Units,Learning_Rate,Dropout,Batch_Size,Best_Epoch,Best_Val_Loss,Val_MSE,Val_RMSE,Val_MAE,Val_R2,Training_Time_s,Parameters
0,10,LSTM,128,0.0005,0.1,16,14,584.667969,584.668002,24.179909,15.961914,0.000016,20.820016,67201
1,3,LSTM,64,0.0010,0.0,32,29,584.673950,584.673995,24.180033,15.915537,0.000006,12.714773,17217
2,6,LSTM,128,0.0010,0.0,32,14,584.681763,584.681724,24.180193,15.909536,-0.000008,14.481534,67201
3,8,LSTM,64,0.0010,0.1,16,15,584.689880,584.689917,24.180362,15.891234,-0.000022,12.070537,17217
4,9,LSTM,64,0.0005,0.1,16,28,584.698364,584.698320,24.180536,16.017350,-0.000036,18.514198,17217
5,7,LSTM,128,0.0005,0.0,32,26,584.739624,584.739692,24.181391,16.059037,-0.000107,19.626857,67201
6,11,LSTM,128,0.0005,0.2,32,28,584.740540,584.740597,24.181410,15.844994,-0.000108,21.084662,67201
7,4,LSTM,64,0.0005,0.0,32,50,596.058655,596.058607,24.414312,17.753658,-0.019466,15.429940,17217
8,1,LSTM,32,0.0010,0.0,32,50,598.451843,598.451879,24.463276,17.969177,-0.023559,12.857429,4513
9,2,LSTM,32,0.0005,0.0,32,50,1206.073975,1206.073986,34.728576,33.276662,-1.062803,12.958996,4513


# EVALUATION METRICS FOR BEST LSTM CONFIGURATION

In [24]:
best_lstm = lstm_df_sorted.iloc[0]

print("BEST LSTM CONFIGURATION")
print("-" * 40)
print(f"Units        : {best_lstm['Units']}")
print(f"Learning Rate: {best_lstm['Learning_Rate']}")
print(f"Dropout      : {best_lstm['Dropout']}")
print(f"Batch Size   : {best_lstm['Batch_Size']}")
print(f"Best Epoch   : {best_lstm['Best_Epoch']}")
print(f"Val RMSE     : {best_lstm['Val_RMSE']:.6f}")
print(f"Val MAE      : {best_lstm['Val_MAE']:.6f}")
print(f"Val R²       : {best_lstm['Val_R2']:.6f}")
print(f"Parameters   : {best_lstm['Parameters']}")
print(f"Training Time: {best_lstm['Training_Time_s']:.2f} s")

BEST LSTM CONFIGURATION
----------------------------------------
Units        : 128
Learning Rate: 0.0005
Dropout      : 0.1
Batch Size   : 16
Best Epoch   : 14
Val RMSE     : 24.179909
Val MAE      : 15.961914
Val R²       : 0.000016
Parameters   : 67201
Training Time: 20.82 s


# GRU MODEL

In [25]:
# ============================================================
# DAY 19 - GRU MODEL
# ============================================================

def build_gru_model(
    units=64,
    dropout=0.0,
    learning_rate=0.001
):

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(X_train.shape[1], X_train.shape[2])
        ),

        # GRU Layer 1
        tf.keras.layers.GRU(
            units,
            return_sequences=True
        ),

        tf.keras.layers.Dropout(dropout),

        # GRU Layer 2
        tf.keras.layers.GRU(
            units,
            return_sequences=False
        ),

        tf.keras.layers.Dropout(dropout),

        # Fully connected layers
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [26]:
# Test GRU model
gru_test_model = build_gru_model(
    units=64,
    dropout=0.2,
    learning_rate=0.001
)

gru_test_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 10, 64)         │        13,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,129 (156.75 KB)

 Trainable params: 40,129 (156.75 KB)

 Non-trainable params: 0 (0.00 B)

In [28]:
# ============================================================
# DAY 19 - CONTROLLED GRU HYPERPARAMETER TUNING
# ============================================================

GRU_EXPERIMENTS = [

    # Units, Learning Rate, Dropout, Batch Size

    (32,  0.0010, 0.0, 32),
    (32,  0.0005, 0.0, 32),

    (64,  0.0010, 0.0, 32),
    (64,  0.0005, 0.0, 32),
    (64,  0.0001, 0.0, 32),

    (128, 0.0010, 0.0, 32),
    (128, 0.0005, 0.0, 32),

    (64,  0.0010, 0.1, 16),
    (64,  0.0005, 0.1, 16),

    (128, 0.0005, 0.1, 16),
    (128, 0.0005, 0.2, 32),
    (128, 0.0001, 0.1, 32)
]

MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8

print("Total GRU experiments:", len(GRU_EXPERIMENTS))

Total GRU experiments: 12


# GRID SEARCH

In [33]:
# ============================================================
# RUN CONTROLLED GRU EXPERIMENTS
# ============================================================

gru_results = []

RESULTS_FILE = "Day19_GRU_Tuning_Results.csv"

for run_no, (units, learning_rate, dropout, batch_size) in enumerate(
    GRU_EXPERIMENTS, start=1
):

    print("\n" + "=" * 70)
    print(f"GRU Experiment {run_no}/{len(GRU_EXPERIMENTS)}")
    print(f"Units        : {units}")
    print(f"Learning Rate: {learning_rate}")
    print(f"Dropout      : {dropout}")
    print(f"Batch Size   : {batch_size}")
    print("=" * 70)

    # --------------------------------------------------------
    # Clear previous TensorFlow model
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    # Reproducibility
    tf.random.set_seed(42)
    np.random.seed(42)

    # --------------------------------------------------------
    # Build fresh GRU model
    # --------------------------------------------------------

    model = build_gru_model(
        units=units,
        dropout=dropout,
        learning_rate=learning_rate
    )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=0
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    start_train = time.time()

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        callbacks=[early_stopping],
        verbose=0
    )

    training_time = time.time() - start_train

    # --------------------------------------------------------
    # Validation prediction
    # --------------------------------------------------------

    start_inference = time.time()

    y_val_pred = model.predict(
        X_val,
        verbose=0
    ).ravel()

    inference_time = time.time() - start_inference

    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------

    val_mse = mean_squared_error(
        y_val,
        y_val_pred
    )

    val_rmse = np.sqrt(val_mse)

    val_mae = mean_absolute_error(
        y_val,
        y_val_pred
    )

    val_r2 = r2_score(
        y_val,
        y_val_pred
    )

    # --------------------------------------------------------
    # Best epoch
    # --------------------------------------------------------

    best_epoch = np.argmin(
        history.history["val_loss"]
    ) + 1

    best_val_loss = np.min(
        history.history["val_loss"]
    )

    # --------------------------------------------------------
    # Number of parameters
    # --------------------------------------------------------

    parameters = model.count_params()

    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    result = {
        "Run": run_no,
        "Model": "GRU",
        "Units": units,
        "Learning_Rate": learning_rate,
        "Dropout": dropout,
        "Batch_Size": batch_size,
        "Best_Epoch": best_epoch,
        "Best_Val_Loss": best_val_loss,
        "Val_MSE": val_mse,
        "Val_RMSE": val_rmse,
        "Val_MAE": val_mae,
        "Val_R2": val_r2,
        "Training_Time_s": training_time,
        "Inference_Time_s": inference_time,
        "Parameters": parameters
    }

    gru_results.append(result)

    # --------------------------------------------------------
    # Save results after every experiment
    # --------------------------------------------------------

    pd.DataFrame(gru_results).to_csv(
        RESULTS_FILE,
        index=False
    )

    print(
        f"Val RMSE = {val_rmse:.6f} | "
        f"Val MAE = {val_mae:.6f} | "
        f"Val R² = {val_r2:.6f} | "
        f"Time = {training_time:.2f}s"
    )

print("\nGRU controlled experiments completed!")
print(f"Total experiments: {len(gru_results)}")
print(f"Results saved to: {RESULTS_FILE}")


GRU Experiment 1/12
Units        : 32
Learning Rate: 0.001
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 24.213484 | Val MAE = 15.503194 | Val R² = -0.002763 | Time = 11.63s

GRU Experiment 2/12
Units        : 32
Learning Rate: 0.0005
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 24.194198 | Val MAE = 15.648997 | Val R² = -0.001166 | Time = 14.14s

GRU Experiment 3/12
Units        : 64
Learning Rate: 0.001
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 24.422439 | Val MAE = 14.887280 | Val R² = -0.020145 | Time = 10.16s

GRU Experiment 4/12
Units        : 64
Learning Rate: 0.0005
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 24.228006 | Val MAE = 15.424028 | Val R² = -0.003966 | Time = 12.06s

GRU Experiment 5/12
Units        : 64
Learning Rate: 0.0001
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 24.180023 | Val MAE = 15.962087 | Val R² = 0.000006 | Time = 30.97s

GRU Experiment 6/12
Units        : 128
Learning Rate: 0.001
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 24.1

# BEST GRU

In [32]:
gru_df = pd.DataFrame(gru_results)

gru_df_sorted = gru_df.sort_values(
    by="Val_RMSE",
    ascending=True
).reset_index(drop=True)

print("TOP 10 GRU CONFIGURATIONS")

display(
    gru_df_sorted.head(10)
)

TOP 10 GRU CONFIGURATIONS


,Run,Model,Units,Learning_Rate,Dropout,Batch_Size,Best_Epoch,Best_Val_Loss,Val_MSE,Val_RMSE,Val_MAE,Val_R2,Training_Time_s,Inference_Time_s,Parameters
0,2,GRU,32,0.0005,0.0,32,11,584.675232,584.675288,24.180060,15.925757,0.000003,14.496299,0.709891,10881
1,12,GRU,128,0.0001,0.1,32,21,584.683350,584.683403,24.180228,15.916199,-0.000010,41.301710,0.623204,153921
2,1,GRU,32,0.0010,0.0,32,6,584.962097,584.962119,24.185990,15.749616,-0.000487,14.284144,0.632255,10881
3,4,GRU,64,0.0005,0.0,32,6,585.042542,585.042534,24.187653,16.205233,-0.000625,13.797173,0.657202,40129
4,5,GRU,64,0.0001,0.0,32,50,585.456238,585.456255,24.196203,16.332337,-0.001332,39.447699,0.759730,40129
5,6,GRU,128,0.0010,0.0,32,3,586.944092,586.944109,24.226929,15.429758,-0.003877,17.156289,1.346111,153921
6,9,GRU,64,0.0005,0.1,16,4,587.105347,587.105385,24.230258,15.413113,-0.004153,18.605729,0.640679,40129
7,10,GRU,128,0.0005,0.1,16,2,587.655640,587.655615,24.241609,16.767402,-0.005094,22.013975,0.620876,153921
8,11,GRU,128,0.0005,0.2,32,6,591.091248,591.091254,24.312368,15.121276,-0.010970,19.582603,0.650404,153921
9,7,GRU,128,0.0005,0.0,32,4,595.913696,595.913698,24.411344,14.906168,-0.019218,17.930143,0.574996,153921


# evaluation metrics

In [35]:
best_gru = gru_df_sorted.iloc[0]

print("\nBEST GRU CONFIGURATION")
print("-" * 40)
print(f"Units        : {best_gru['Units']}")
print(f"Learning Rate: {best_gru['Learning_Rate']}")
print(f"Dropout      : {best_gru['Dropout']}")
print(f"Batch Size   : {best_gru['Batch_Size']}")
print(f"Best Epoch   : {best_gru['Best_Epoch']}")
print(f"Val RMSE     : {best_gru['Val_RMSE']:.6f}")
print(f"Val MAE      : {best_gru['Val_MAE']:.6f}")
print(f"Val R²       : {best_gru['Val_R2']:.6f}")
print(f"Parameters   : {best_gru['Parameters']}")
print(f"Training Time: {best_gru['Training_Time_s']:.2f} s")


BEST GRU CONFIGURATION
----------------------------------------
Units        : 32
Learning Rate: 0.0005
Dropout      : 0.0
Batch Size   : 32
Best Epoch   : 11
Val RMSE     : 24.180060
Val MAE      : 15.925757
Val R²       : 0.000003
Parameters   : 10881
Training Time: 14.50 s


# TCN

In [42]:
def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout):

    # First causal convolution
    conv1 = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)

    conv1 = tf.keras.layers.Dropout(dropout)(conv1)

    # Second causal convolution
    conv2 = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(conv1)

    conv2 = tf.keras.layers.Dropout(dropout)(conv2)

    # Residual connection
    if x.shape[-1] != filters:
        residual = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=1,
            padding="same"
        )(x)
    else:
        residual = x

    output = tf.keras.layers.Add()([residual, conv2])
    output = tf.keras.layers.ReLU()(output)

    return output


def build_tcn_model(
    units=64,
    dropout=0.0,
    learning_rate=0.001
):

    inputs = tf.keras.layers.Input(
        shape=(X_train.shape[1], X_train.shape[2])
    )

    x = inputs

    # TCN residual blocks
    for dilation_rate in [1, 2, 4]:

        x = tcn_residual_block(
            x=x,
            filters=units,
            kernel_size=3,
            dilation_rate=dilation_rate,
            dropout=dropout
        )

    # Take final time step
    x = tf.keras.layers.Lambda(
        lambda tensor: tensor[:, -1, :]
    )(x)

    # Fully connected layers
    x = tf.keras.layers.Dense(
        32,
        activation="relu"
    )(x)

    outputs = tf.keras.layers.Dense(1)(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [43]:
# ============================================================
# TEST TCN MODEL
# ============================================================

tcn_test_model = build_tcn_model(
    units=64,
    dropout=0.2,
    learning_rate=0.001
)

tcn_test_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 10, 2)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 10, 64)    │        448 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 10, 64)    │          0 │ conv1d_7[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 10, 64)    │     12,352 │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 10, 64)    │        192 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 10, 64)    │          0 │ conv1d_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 10, 64)    │          0 │ conv1d_9[0][0],   │
│                     │                   │            │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 10, 64)    │          0 │ add_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_10 (Conv1D)  │ (None, 10, 64)    │     12,352 │ re_lu_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 10, 64)    │          0 │ conv1d_10[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (None, 10, 64)    │     12,352 │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 10, 64)    │          0 │ conv1d_11[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 10, 64)    │          0 │ re_lu_3[0][0],    │
│                     │                   │            │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_4 (ReLU)      │ (None, 10, 64)    │          0 │ add_4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_12 (Conv1D)  │ (None, 10, 64)    │     12,352 │ re_lu_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 10, 64)    │          0 │ conv1d_12[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 10, 64)    │     12,352 │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 10, 64)    │          0 │ conv1d_13[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 10, 64)    │          0 │ re_lu_4[0][0],    │
│                     │                   │            │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_5 (ReLU)      │ (None, 10, 64)    │          0 │ add_5[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 64)        │          0 │ re_lu_5[0][0]   

 Total params: 64,513 (252.00 KB)

 Trainable params: 64,513 (252.00 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
# ============================================================
# DAY 19 - TCN HYPERPARAMETER SEARCH
# EXACTLY 200 EXPERIMENTS
# ============================================================

import itertools

TCN_UNITS_RANGE = [16, 32, 64, 128, 256]

TCN_LEARNING_RATE_RANGE = [
    0.00001,
    0.00005,
    0.0001,
    0.0005,
    0.001
]

TCN_DROPOUT_RANGE = [
    0.0,
    0.1,
    0.2,
    0.3
]

TCN_BATCH_SIZE_RANGE = [
    16,
    32
]

# ------------------------------------------------------------
# Generate exactly 200 combinations
# ------------------------------------------------------------

tcn_combinations = list(itertools.product(
    TCN_UNITS_RANGE,
    TCN_LEARNING_RATE_RANGE,
    TCN_DROPOUT_RANGE,
    TCN_BATCH_SIZE_RANGE
))

print("Total TCN experiments:", len(tcn_combinations))

Total TCN experiments: 200


# Grid Search

In [45]:
# ============================================================
# RUN TCN - 200 EXPERIMENTS
# ============================================================

tcn_results = []

RESULTS_FILE = "Day19_TCN_Tuning_Results.csv"

for run_no, (units, learning_rate, dropout, batch_size) in enumerate(
    tcn_combinations, start=1
):

    print("\n" + "=" * 70)
    print(f"TCN Experiment {run_no}/{len(tcn_combinations)}")
    print(f"Units        : {units}")
    print(f"Learning Rate: {learning_rate}")
    print(f"Dropout      : {dropout}")
    print(f"Batch Size   : {batch_size}")
    print("=" * 70)

    tf.keras.backend.clear_session()

    tf.random.set_seed(42)
    np.random.seed(42)

    model = build_tcn_model(
        units=units,
        dropout=dropout,
        learning_rate=learning_rate
    )

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=0
    )

    start_train = time.time()

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        callbacks=[early_stopping],
        verbose=0
    )

    training_time = time.time() - start_train

    y_val_pred = model.predict(
        X_val,
        verbose=0
    ).ravel()

    val_mse = mean_squared_error(
        y_val,
        y_val_pred
    )

    val_rmse = np.sqrt(val_mse)

    val_mae = mean_absolute_error(
        y_val,
        y_val_pred
    )

    val_r2 = r2_score(
        y_val,
        y_val_pred
    )

    best_epoch = np.argmin(
        history.history["val_loss"]
    ) + 1

    best_val_loss = np.min(
        history.history["val_loss"]
    )

    tcn_results.append({
        "Run": run_no,
        "Model": "TCN",
        "Units": units,
        "Learning_Rate": learning_rate,
        "Dropout": dropout,
        "Batch_Size": batch_size,
        "Best_Epoch": best_epoch,
        "Best_Val_Loss": best_val_loss,
        "Val_MSE": val_mse,
        "Val_RMSE": val_rmse,
        "Val_MAE": val_mae,
        "Val_R2": val_r2,
        "Training_Time_s": training_time,
        "Parameters": model.count_params()
    })

    pd.DataFrame(tcn_results).to_csv(
        RESULTS_FILE,
        index=False
    )

    print(
        f"Val RMSE = {val_rmse:.6f} | "
        f"Val MAE = {val_mae:.6f} | "
        f"Val R² = {val_r2:.6f}"
    )

print("\nTCN 200-experiment search completed!")
print(f"Experiments completed: {len(tcn_results)}")


TCN Experiment 1/200
Units        : 16
Learning Rate: 1e-05
Dropout      : 0.0
Batch Size   : 16
Val RMSE = 22.381173 | Val MAE = 14.606470 | Val R² = 0.143259

TCN Experiment 2/200
Units        : 16
Learning Rate: 1e-05
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 65.793387 | Val MAE = 61.697255 | Val R² = -6.403690

TCN Experiment 3/200
Units        : 16
Learning Rate: 1e-05
Dropout      : 0.1
Batch Size   : 16
Val RMSE = 22.731541 | Val MAE = 16.620720 | Val R² = 0.116225

TCN Experiment 4/200
Units        : 16
Learning Rate: 1e-05
Dropout      : 0.1
Batch Size   : 32
Val RMSE = 70.685240 | Val MAE = 66.465643 | Val R² = -7.545574

TCN Experiment 5/200
Units        : 16
Learning Rate: 1e-05
Dropout      : 0.2
Batch Size   : 16
Val RMSE = 22.338000 | Val MAE = 15.838150 | Val R² = 0.146561

TCN Experiment 6/200
Units        : 16
Learning Rate: 1e-05
Dropout      : 0.2
Batch Size   : 32
Val RMSE = 64.013096 | Val MAE = 60.098385 | Val R² = -6.008441

TCN Experiment 7/200
Units    

KeyboardInterrupt: 

# Best TCN Configuration

In [47]:
# ============================================================
# BEST TCN CONFIGURATION
# ============================================================

tcn_df = pd.DataFrame(tcn_results)

tcn_df_sorted = tcn_df.sort_values(
    by="Val_RMSE",
    ascending=True
).reset_index(drop=True)

print("TOP 10 TCN CONFIGURATIONS")

display(
    tcn_df_sorted.head(10)
)

TOP 10 TCN CONFIGURATIONS


,Run,Model,Units,Learning_Rate,Dropout,Batch_Size,Best_Epoch,Best_Val_Loss,Val_MSE,Val_RMSE,Val_MAE,Val_R2,Training_Time_s,Parameters
0,54,TCN,32,0.00005,0.2,32,28,478.769531,478.769506,21.880802,14.343437,0.181139,19.676855,16929
1,119,TCN,64,0.00100,0.3,16,1,481.482300,481.482323,21.942705,14.314913,0.176499,9.532005,64513
2,136,TCN,128,0.00005,0.3,32,12,482.138855,482.138815,21.957660,15.039913,0.175376,25.586584,251841
3,93,TCN,64,0.00005,0.2,16,11,483.703979,483.703928,21.993270,14.470820,0.172699,16.949589,64513
4,134,TCN,128,0.00005,0.2,32,11,483.876678,483.876709,21.997198,14.673888,0.172404,23.297655,251841
5,95,TCN,64,0.00005,0.3,16,13,483.921265,483.921254,21.998210,14.933799,0.172328,19.127424,64513
6,135,TCN,128,0.00005,0.3,16,8,484.211670,484.211635,22.004809,14.568526,0.171831,24.044222,251841
7,128,TCN,128,0.00001,0.3,32,12,484.699585,484.699611,22.015895,15.445580,0.170996,24.042099,251841
8,96,TCN,64,0.00005,0.3,32,18,484.935394,484.935418,22.021249,14.967352,0.170593,18.258488,64513
9,120,TCN,64,0.00100,0.3,32,2,485.115143,485.115199,22.025331,14.265567,0.170286,8.524899,64513


# Evaluation Metrics

In [48]:
best_tcn = tcn_df_sorted.iloc[0]

print("\nBEST TCN CONFIGURATION")
print("-" * 40)
print(f"Filters/Units : {best_tcn['Units']}")
print(f"Learning Rate : {best_tcn['Learning_Rate']}")
print(f"Dropout       : {best_tcn['Dropout']}")
print(f"Batch Size    : {best_tcn['Batch_Size']}")
print(f"Best Epoch    : {best_tcn['Best_Epoch']}")
print(f"Val RMSE      : {best_tcn['Val_RMSE']:.6f}")
print(f"Val MAE       : {best_tcn['Val_MAE']:.6f}")
print(f"Val R²        : {best_tcn['Val_R2']:.6f}")
print(f"Parameters    : {best_tcn['Parameters']}")
print(f"Training Time : {best_tcn['Training_Time_s']:.2f} s")


BEST TCN CONFIGURATION
----------------------------------------
Filters/Units : 32
Learning Rate : 5e-05
Dropout       : 0.2
Batch Size    : 32
Best Epoch    : 28
Val RMSE      : 21.880802
Val MAE       : 14.343437
Val R²        : 0.181139
Parameters    : 16929
Training Time : 19.68 s
